# Building the flights tables with MobilitySpark

This notebook builds the flights data of the book from the state vectors that the
[OpenSky Network](https://opensky-network.org/) publishes for **Sunday, March 1, 2026**, as a
sample of the table `state_vectors` of its historical database. It is the Spark counterpart of the
script `opensky_input.sql` of chapter 5 of the first volume, which builds the same tables in
MobilityDB for June 1, 2020, and the statements below follow that script one for one:

| Table | Content |
|---|---|
| `FlightsInput` | The state vectors, cleaned, with their timestamps and their position |
| `FlightsPositions` | The position reports, once each at its time, that the aircraft can reach |
| `FlightsDay` | One row per aircraft, its whole day as temporal values |
| `Flights` | One row per flight, i.e., per aircraft and call sign |

The temporal types and their functions come from **MobilitySpark**, whose typed SQL surface is
generated from the MEOS library: `tgeogpoint`, `tfloat`, `ttext` and `tbool` are Spark types, and
the functions keep their MobilityDB names (`tgeogpointSeq`, `atTime`, ...), except where Spark or
Flink owns the name, as `temporalUnnest` for `unnest`. The points
are built with Apache Sedona, and the tables are stored in Delta Lake.

The day is downloaded by `flights/fetch_data.sh` of this repository into
`/opt/data/flights/parquet`, 24 Parquet files of one hour each, in the folders
`hour=<Unix time of the hour>`. The appendix "The Flights Data of the Book" describes the data and
the rules applied below.

In [1]:
## Run this cell FIRST

import subprocess
import time
import sys

print("Starting Safe YARN Cluster Cleanup...")

# 1. Clean up local zombie spark references before querying YARN
if 'spark' in sys.modules or 'pyspark' in sys.modules:
    try:
        from pyspark.sql import SparkSession
        active_session = SparkSession.getActiveSession()
        if active_session:
            print("Stopping local legacy session safely...")
            active_session.stop()
    except Exception:
        pass

try:
    # 2. Query YARN for active sessions
    cmd_list = "yarn application -list -appStates RUNNING,ACCEPTED"
    result = subprocess.check_output(cmd_list, shell=True, text=True)
    
    lines = result.strip().split("\n")
    apps_killed = 0
    
    for line in lines:
        if "application_" in line:
            app_id = line.split()[0]
            print(f"Killing conflicting YARN session: {app_id}")
            subprocess.call(f"yarn application -kill {app_id}", shell=True)
            apps_killed += 1
            
    if apps_killed > 0:
        print("Waiting for resource managers to clear memory allocations...")
        time.sleep(6)
        print("Cluster environment successfully normalized!")
    else:
        print("Cluster queue is already wide open and clean.")

except Exception as e:
    print(f"YARN query bypassed: {e}")


Starting Safe YARN Cluster Cleanup...
Cluster queue is already wide open and clean.


In [2]:
## Run this cell SECOND

import os
from pyspark.sql import SparkSession

print("Allocating new Spark Session...")

# The master is YARN in the cluster of the chapter; SPARK_MASTER=local[*] runs the notebook on
# a single machine. The session time zone is UTC, the time zone of the OpenSky timestamps.
spark = SparkSession.builder \
    .appName("MobilitySpark-Flights") \
    .config("spark.master", os.environ.get("SPARK_MASTER", "yarn")) \
    .config("spark.submit.deployMode", "client") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "2g")) \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

# Register the typed SQL surface of MobilitySpark: each MobilityDB type (tgeogpoint, tfloat,
# ttext, ...) is a Spark type, and each MobilityDB function is called by its MobilityDB name
spark._jvm.org.mobilitydb.spark.sql.MobilitySparkSql.registerAll(spark._jsparkSession)

print("Spark Session successfully attached to " + spark.sparkContext.master)

Allocating new Spark Session...
Spark Session successfully attached to local[6]


In [3]:
## Run this cell THIRD

from pyspark.sql import SparkSession
from IPython.core.magic import register_cell_magic


@register_cell_magic
def sql(line, cell):

    # Eliminar comentarios SQL de líneas completas
    lines = []
    for line in cell.splitlines():
        if not line.strip().startswith("#"):
            lines.append(line)

    cell = "\n".join(lines)

    # Separar sentencias
    statements = [
        s.strip()
        for s in cell.split(";")
        if s.strip()
    ]

    for statement in statements:
        print("\n==============================")
        print("RUNNING:")
        print(statement)
        print("==============================")

        result = spark.sql(statement)
        # Only attempt to call show() if the query actually returns data rows
        if result.schema:
            result.show(truncate=False)
        else:
            print("Command executed successfully.")
            

print("cell magic %%sql prepared")

cell magic %%sql prepared


In [4]:
## Run this cell FOURTH

# Remove the folders of the four tables of this notebook from the warehouse, in case a
# previous run left them without their entry in the catalog. The other tables of the
# warehouse, such as those of the Citi Bike data, are kept.

from py4j.java_gateway import java_import

java_import(spark._jvm, "org.apache.hadoop.fs.Path")

warehouse = spark.conf.get("spark.sql.warehouse.dir")
for table in ["flightsinput", "flightspositions", "flightsday", "flights"]:
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    path = spark._jvm.Path(warehouse + "/" + table)
    path.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration()).delete(path, True)

print("flights tables removed from the warehouse")

flights tables removed from the warehouse


## The state vectors

A state vector reports, for one aircraft identified by its transponder address `ICAO24`, its
position, velocity, heading, vertical rate, call sign, and altitudes at one instant, given as a
Unix timestamp. The files hold one state vector per aircraft and per second, with the columns of
the table `state_vectors`. The view `OpenSky` keeps the state vectors at every tenth second,
`Time % 10 = 0`, the time step of the public samples of OpenSky, and reads the squawk, which the
files store as text, as an integer.

In [5]:
%%sql

CREATE OR REPLACE TEMP VIEW OpenSkyParquet
USING parquet OPTIONS (path 'file:///opt/data/flights/parquet');

CREATE OR REPLACE TEMP VIEW OpenSky AS
SELECT Time, ICAO24, Lat, Lon, Velocity, Heading, VertRate, CallSign, OnGround, Alert, SPI,
  CAST(Squawk AS INT) AS Squawk, BaroAltitude, GeoAltitude, LastPosUpdate, LastContact
FROM OpenSkyParquet
WHERE Time % 10 = 0;

SELECT * FROM OpenSky LIMIT 5;


RUNNING:
CREATE OR REPLACE TEMP VIEW OpenSkyParquet
USING parquet OPTIONS (path 'file:///opt/data/flights/parquet')
Command executed successfully.

RUNNING:
CREATE OR REPLACE TEMP VIEW OpenSky AS
SELECT Time, ICAO24, Lat, Lon, Velocity, Heading, VertRate, CallSign, OnGround, Alert, SPI,
  CAST(Squawk AS INT) AS Squawk, BaroAltitude, GeoAltitude, LastPosUpdate, LastContact
FROM OpenSkyParquet
WHERE Time % 10 = 0
Command executed successfully.

RUNNING:
SELECT * FROM OpenSky LIMIT 5
+----------+------+------------------+-----------------+------------------+-----------------+--------+--------+--------+-----+-----+------+------------+-----------+----------------+----------------+
|Time      |ICAO24|Lat               |Lon              |Velocity          |Heading          |VertRate|CallSign|OnGround|Alert|SPI  |Squawk|BaroAltitude|GeoAltitude|LastPosUpdate   |LastContact     |
+----------+------+------------------+-----------------+------------------+-----------------+--------+--------+----

## Cleaning

The table `FlightsInput` keeps the state vectors that have a squawk, the transponder code that
identifies an aircraft on radar, and drops the aircraft that never report a position. It converts
the Unix timestamps into timestamps and builds the position as a Sedona point in WGS84.

In [6]:
%%sql

DROP TABLE IF EXISTS FlightsInput;

CREATE TABLE FlightsInput USING DELTA AS
WITH ICAO24_WithNullLon AS (
  SELECT ICAO24 FROM OpenSky GROUP BY ICAO24 HAVING count(Lon) = 0 )
SELECT Time AS Et, ICAO24, Lat, Lon, Velocity, Heading, VertRate, CallSign, OnGround,
  Alert, SPI, Squawk, BaroAltitude, GeoAltitude, LastPosUpdate, LastContact,
  timestamp_seconds(Time) AS EtTs,
  timestamp_seconds(LastPosUpdate) AS LastPosUpdateTs,
  timestamp_seconds(LastContact) AS LastContactTs,
  ST_SetSRID(ST_Point(Lon, Lat), 4326) AS Geo
FROM OpenSky
WHERE Squawk IS NOT NULL
  AND ICAO24 NOT IN (SELECT ICAO24 FROM ICAO24_WithNullLon);

SELECT count(*) AS States, count(DISTINCT ICAO24) AS Aircraft FROM FlightsInput;


RUNNING:
DROP TABLE IF EXISTS FlightsInput
Command executed successfully.

RUNNING:
CREATE TABLE FlightsInput USING DELTA AS
WITH ICAO24_WithNullLon AS (
  SELECT ICAO24 FROM OpenSky GROUP BY ICAO24 HAVING count(Lon) = 0 )
SELECT Time AS Et, ICAO24, Lat, Lon, Velocity, Heading, VertRate, CallSign, OnGround,
  Alert, SPI, Squawk, BaroAltitude, GeoAltitude, LastPosUpdate, LastContact,
  timestamp_seconds(Time) AS EtTs,
  timestamp_seconds(LastPosUpdate) AS LastPosUpdateTs,
  timestamp_seconds(LastContact) AS LastContactTs,
  ST_SetSRID(ST_Point(Lon, Lat), 4326) AS Geo
FROM OpenSky
WHERE Squawk IS NOT NULL
  AND ICAO24 NOT IN (SELECT ICAO24 FROM ICAO24_WithNullLon)
Command executed successfully.

RUNNING:
SELECT count(*) AS States, count(DISTINCT ICAO24) AS Aircraft FROM FlightsInput
+--------+--------+
|States  |Aircraft|
+--------+--------+
|34067894|29395   |
+--------+--------+



## The position reports

A state vector repeats the last position of its aircraft until a new position report arrives, and
`LastPosUpdate` is the time of that report. The table `FlightsPositions` takes each report once,
at its time, from the first state vector that carries it, and keeps the reports that the aircraft
can reach, by the walk of `ais/ais_clean.sql` with the speed of an aircraft for that of a vessel:

* A report is kept when the last kept report could reach it at 400 m/s along the great circle,
  the distance being measured on the WGS84 spheroid by `ST_DistanceSpheroid`.
* The walk trusts its first report, so the seed is corroborated: it is the earliest of the first
  eleven reports of an aircraft that a majority of them can reach. The reports before the seed
  are walked backwards from it by the same rule.
* An aircraft whose share of reports unreachable from the report before exceeds one half is
  dropped whole.

Spark SQL has no recursive query, and the walk of an aircraft is one pass over its reports in the
order of their time: the higher-order function `aggregate` makes that pass over
`array_sort(collect_list(struct(...)))`, carrying the last kept report, the anchor, and the numbers
of the reports kept so far. `CACHE TABLE` keeps the intermediate tables in memory for the
statement that follows.

In [7]:
%%sql

CACHE TABLE Reports AS
SELECT ICAO24, row_number() OVER (PARTITION BY ICAO24 ORDER BY LastPosUpdate) AS Rn,
  LastPosUpdate AS Ts, LastPosUpdateTs, Lon, Lat, GeoAltitude, Geo
FROM (
  SELECT ICAO24, LastPosUpdate, LastPosUpdateTs, Lon, Lat, GeoAltitude, Geo,
    row_number() OVER (PARTITION BY ICAO24, LastPosUpdate ORDER BY Et) AS RowNo
  FROM FlightsInput
  WHERE Geo IS NOT NULL AND LastPosUpdate IS NOT NULL ) p
WHERE RowNo = 1;

CACHE TABLE KeepAircraft AS
SELECT ICAO24 FROM (
  SELECT ICAO24, NOT coalesce(ST_DistanceSpheroid(lag(Geo) OVER w, Geo) <=
    400 * (Ts - lag(Ts) OVER w), true) AS Unreachable
  FROM Reports WINDOW w AS (PARTITION BY ICAO24 ORDER BY Rn)) s
GROUP BY ICAO24 HAVING avg(CAST(Unreachable AS INT)) <= 0.5;

CACHE TABLE Head AS
SELECT r.ICAO24, r.Rn, r.Ts, r.Geo
FROM Reports r JOIN KeepAircraft k ON k.ICAO24 = r.ICAO24
WHERE r.Rn <= 11;

CACHE TABLE Seed AS
SELECT ICAO24, coalesce(min(Rn) FILTER (WHERE Ok * 2 >= Others), 1) AS SeedRn
FROM (
  SELECT a.ICAO24, a.Rn,
    count(*) FILTER (WHERE b.Rn <> a.Rn AND b.Ts <> a.Ts
      AND ST_DistanceSpheroid(a.Geo, b.Geo) <= 400 * abs(b.Ts - a.Ts)) AS Ok,
    count(*) - 1 AS Others
  FROM Head a JOIN Head b ON b.ICAO24 = a.ICAO24
  GROUP BY a.ICAO24, a.Rn) a
GROUP BY ICAO24;

DROP TABLE IF EXISTS FlightsPositions;

CREATE TABLE FlightsPositions USING DELTA AS
WITH Tracks AS (
  SELECT r.ICAO24, s.SeedRn,
    array_sort(collect_list(struct(r.Rn, r.Ts, r.Lon, r.Lat))) AS Reports
  FROM Reports r JOIN Seed s ON s.ICAO24 = r.ICAO24
  GROUP BY r.ICAO24, s.SeedRn ),
Walks AS (
  SELECT ICAO24,
    aggregate(filter(Reports, x -> x.Rn > SeedRn),
      named_struct('Ts', Reports[SeedRn - 1].Ts, 'Lon', Reports[SeedRn - 1].Lon,
        'Lat', Reports[SeedRn - 1].Lat, 'Kept', array(SeedRn)),
      (a, x) -> CASE
        WHEN ST_DistanceSpheroid(ST_Point(a.Lon, a.Lat), ST_Point(x.Lon, x.Lat)) <=
          400 * (x.Ts - a.Ts)
        THEN named_struct('Ts', x.Ts, 'Lon', x.Lon, 'Lat', x.Lat,
          'Kept', array_append(a.Kept, x.Rn))
        ELSE a END).Kept AS Forward,
    aggregate(reverse(filter(Reports, x -> x.Rn < SeedRn)),
      named_struct('Ts', Reports[SeedRn - 1].Ts, 'Lon', Reports[SeedRn - 1].Lon,
        'Lat', Reports[SeedRn - 1].Lat, 'Kept', CAST(array() AS ARRAY<INT>)),
      (a, x) -> CASE
        WHEN ST_DistanceSpheroid(ST_Point(a.Lon, a.Lat), ST_Point(x.Lon, x.Lat)) <=
          400 * (a.Ts - x.Ts)
        THEN named_struct('Ts', x.Ts, 'Lon', x.Lon, 'Lat', x.Lat,
          'Kept', array_append(a.Kept, x.Rn))
        ELSE a END).Kept AS Backward
  FROM Tracks ),
Kept AS (
  SELECT ICAO24, explode(concat(Forward, Backward)) AS Rn
  FROM Walks )
SELECT r.ICAO24, r.LastPosUpdateTs AS PosTs, r.Lon, r.Lat, r.GeoAltitude, r.Geo
FROM Reports r JOIN Kept k ON k.ICAO24 = r.ICAO24 AND k.Rn = r.Rn;

SELECT (SELECT count(*) FROM Reports) AS Reports, count(*) AS KeptReports,
  count(DISTINCT ICAO24) AS Aircraft
FROM FlightsPositions;


RUNNING:
CACHE TABLE Reports AS
SELECT ICAO24, row_number() OVER (PARTITION BY ICAO24 ORDER BY LastPosUpdate) AS Rn,
  LastPosUpdate AS Ts, LastPosUpdateTs, Lon, Lat, GeoAltitude, Geo
FROM (
  SELECT ICAO24, LastPosUpdate, LastPosUpdateTs, Lon, Lat, GeoAltitude, Geo,
    row_number() OVER (PARTITION BY ICAO24, LastPosUpdate ORDER BY Et) AS RowNo
  FROM FlightsInput
  WHERE Geo IS NOT NULL AND LastPosUpdate IS NOT NULL ) p
WHERE RowNo = 1
Command executed successfully.

RUNNING:
CACHE TABLE KeepAircraft AS
SELECT ICAO24 FROM (
  SELECT ICAO24, NOT coalesce(ST_DistanceSpheroid(lag(Geo) OVER w, Geo) <=
    400 * (Ts - lag(Ts) OVER w), true) AS Unreachable
  FROM Reports WINDOW w AS (PARTITION BY ICAO24 ORDER BY Rn)) s
GROUP BY ICAO24 HAVING avg(CAST(Unreachable AS INT)) <= 0.5
Command executed successfully.

RUNNING:
CACHE TABLE Head AS
SELECT r.ICAO24, r.Rn, r.Ts, r.Geo
FROM Reports r JOIN KeepAircraft k ON k.ICAO24 = r.ICAO24
WHERE r.Rn <= 11
Command executed successfully.

RUNNING:
CA

## The day of each aircraft

The table `FlightsDay` holds, for each aircraft, its trajectory as a `tgeogpoint` and its
velocity, heading, vertical rate, call sign, alert flag, and geometric altitude as temporal values.
As in MobilityDB, an instant is made of a value and a timestamp, the instants of an aircraft are
gathered in the order of their timestamps, and a sequence is built from them:

* The trajectory is made of the position reports of `FlightsPositions`, at the times of the
  reports. The other attributes keep the time of their state vector.
* Sedona builds the point and MEOS reads it, as MobilityDB reads a PostGIS point: Sedona writes
  it as hexadecimal extended well-known binary, which keeps its SRID, and
  `geographyFromHexEWKB` reads it as a geography, `geographyFromHexEWKB(ST_AsHEXEWKB(Geo))`.
* Spark has no ordered `array_agg`: `array_sort(collect_list(struct(EtTs, Value)))` orders the
  pairs by their timestamp, and `.Value` keeps their values in that order.
* For an aircraft that never reports an attribute, `collect_list` answers an empty array where
  `array_agg` in PostgreSQL answers `NULL`. Since a sequence needs at least one instant, a `CASE`
  keeps such an attribute `NULL`. An aircraft without a kept position report has no row in
  `Trajectories`, and the `LEFT JOIN` keeps its trajectory `NULL`.

In [8]:
%%sql

DROP TABLE IF EXISTS FlightsDay;

CREATE TABLE FlightsDay USING DELTA AS
WITH Trajectories AS (
  SELECT ICAO24,
    tgeogpointSeq(array_sort(collect_list(struct(PosTs,
      tgeogpoint(geographyFromHexEWKB(ST_AsHEXEWKB(Geo)), PosTs) AS Geo))).Geo) AS Flight
  FROM FlightsPositions
  GROUP BY ICAO24 ),
Instants AS (
  SELECT ICAO24, EtTs,
    tfloat(Velocity, EtTs) AS Velocity, tfloat(Heading, EtTs) AS Heading,
    tfloat(VertRate, EtTs) AS VertRate, ttext(trim(CallSign), EtTs) AS CallSign,
    tbool(Alert, EtTs) AS Alert, tfloat(GeoAltitude, EtTs) AS GeoAltitude
  FROM FlightsInput ),
Arrays AS (
  SELECT ICAO24,
    array_sort(collect_list(struct(EtTs, Velocity)) FILTER (WHERE Velocity IS NOT NULL)).Velocity AS Velocity,
    array_sort(collect_list(struct(EtTs, Heading)) FILTER (WHERE Heading IS NOT NULL)).Heading AS Heading,
    array_sort(collect_list(struct(EtTs, VertRate)) FILTER (WHERE VertRate IS NOT NULL)).VertRate AS VertRate,
    array_sort(collect_list(struct(EtTs, CallSign)) FILTER (WHERE CallSign IS NOT NULL)).CallSign AS CallSign,
    array_sort(collect_list(struct(EtTs, Alert)) FILTER (WHERE Alert IS NOT NULL)).Alert AS Alert,
    array_sort(collect_list(struct(EtTs, GeoAltitude)) FILTER (WHERE GeoAltitude IS NOT NULL)).GeoAltitude AS GeoAltitude
  FROM Instants
  GROUP BY ICAO24 ),
Attributes AS (
  SELECT ICAO24,
    CASE WHEN size(Velocity) > 0 THEN tfloatSeq(Velocity) END AS Velocity,
    CASE WHEN size(Heading) > 0 THEN tfloatSeq(Heading) END AS Heading,
    CASE WHEN size(VertRate) > 0 THEN tfloatSeq(VertRate) END AS VertRate,
    CASE WHEN size(CallSign) > 0 THEN ttextSeq(CallSign) END AS CallSign,
    CASE WHEN size(Alert) > 0 THEN tboolSeq(Alert) END AS Alert,
    CASE WHEN size(GeoAltitude) > 0 THEN tfloatSeq(GeoAltitude) END AS GeoAltitude
  FROM Arrays )
SELECT a.ICAO24, t.Flight, a.Velocity, a.Heading, a.VertRate, a.CallSign, a.Alert,
  a.GeoAltitude
FROM Attributes a LEFT JOIN Trajectories t ON t.ICAO24 = a.ICAO24;

SELECT ICAO24, numInstants(Flight) AS Instants, asText(CallSign) AS CallSign
FROM FlightsDay LIMIT 5;


RUNNING:
DROP TABLE IF EXISTS FlightsDay
Command executed successfully.

RUNNING:
CREATE TABLE FlightsDay USING DELTA AS
WITH Trajectories AS (
  SELECT ICAO24,
    tgeogpointSeq(array_sort(collect_list(struct(PosTs,
      tgeogpoint(geographyFromHexEWKB(ST_AsHEXEWKB(Geo)), PosTs) AS Geo))).Geo) AS Flight
  FROM FlightsPositions
  GROUP BY ICAO24 ),
Instants AS (
  SELECT ICAO24, EtTs,
    tfloat(Velocity, EtTs) AS Velocity, tfloat(Heading, EtTs) AS Heading,
    tfloat(VertRate, EtTs) AS VertRate, ttext(trim(CallSign), EtTs) AS CallSign,
    tbool(Alert, EtTs) AS Alert, tfloat(GeoAltitude, EtTs) AS GeoAltitude
  FROM FlightsInput ),
Arrays AS (
  SELECT ICAO24,
    array_sort(collect_list(struct(EtTs, Velocity)) FILTER (WHERE Velocity IS NOT NULL)).Velocity AS Velocity,
    array_sort(collect_list(struct(EtTs, Heading)) FILTER (WHERE Heading IS NOT NULL)).Heading AS Heading,
    array_sort(collect_list(struct(EtTs, VertRate)) FILTER (WHERE VertRate IS NOT NULL)).VertRate AS VertRate,


## The flights

An aircraft flies several flights in a day, each under its own call sign. The function
`temporalUnnest`, the `unnest` of MobilityDB, splits the temporal call sign of an aircraft into one
pair (value, time) per distinct call sign, where the time is the set of periods during which the
aircraft uses it, and `inline` turns the pairs into rows. The function `atTime` then restricts each
temporal attribute of the aircraft to the periods of the flight.

In [9]:
%%sql

DROP TABLE IF EXISTS Flights;

CREATE TABLE Flights USING DELTA AS
SELECT f.ICAO24, r.Value AS CallSign, r.Time AS FlightPeriod,
  atTime(f.Flight, r.Time) AS Flight, atTime(f.Velocity, r.Time) AS Velocity,
  atTime(f.Heading, r.Time) AS Heading, atTime(f.VertRate, r.Time) AS VertRate,
  atTime(f.Alert, r.Time) AS Alert, atTime(f.GeoAltitude, r.Time) AS GeoAltitude
FROM FlightsDay f
LATERAL VIEW inline(temporalUnnest(f.CallSign)) r AS Value, Time;

SELECT ICAO24, CallSign, asText(FlightPeriod) AS FlightPeriod,
  numInstants(Flight) AS Instants
FROM Flights LIMIT 5;


RUNNING:
DROP TABLE IF EXISTS Flights
Command executed successfully.

RUNNING:
CREATE TABLE Flights USING DELTA AS
SELECT f.ICAO24, r.Value AS CallSign, r.Time AS FlightPeriod,
  atTime(f.Flight, r.Time) AS Flight, atTime(f.Velocity, r.Time) AS Velocity,
  atTime(f.Heading, r.Time) AS Heading, atTime(f.VertRate, r.Time) AS VertRate,
  atTime(f.Alert, r.Time) AS Alert, atTime(f.GeoAltitude, r.Time) AS GeoAltitude
FROM FlightsDay f
LATERAL VIEW inline(temporalUnnest(f.CallSign)) r AS Value, Time
Command executed successfully.

RUNNING:
SELECT ICAO24, CallSign, asText(FlightPeriod) AS FlightPeriod,
  numInstants(Flight) AS Instants
FROM Flights LIMIT 5
+------+--------+--------------------------------------------------+--------+
|ICAO24|CallSign|FlightPeriod                                      |Instants|
+------+--------+--------------------------------------------------+--------+
|01024c|MSC2916 |{[2026-03-01 21:09:00+00, 2026-03-01 23:59:10+00]}|807     |
|01024c|MSC2976 |{[2026-03-01

## Summary of the tables

In [10]:
%%sql

SELECT count(*) AS Aircraft, count(Flight) AS WithFlight,
  sum(numInstants(Flight)) AS Instants
FROM FlightsDay;

SELECT count(*) AS Flights, count(DISTINCT ICAO24) AS Aircraft,
  count(DISTINCT CallSign) AS CallSigns,
  count(*) FILTER (WHERE CallSign = '') AS WithoutCallSign
FROM Flights;


RUNNING:
SELECT count(*) AS Aircraft, count(Flight) AS WithFlight,
  sum(numInstants(Flight)) AS Instants
FROM FlightsDay
+--------+----------+--------+
|Aircraft|WithFlight|Instants|
+--------+----------+--------+
|29395   |29182     |27194486|
+--------+----------+--------+


RUNNING:
SELECT count(*) AS Flights, count(DISTINCT ICAO24) AS Aircraft,
  count(DISTINCT CallSign) AS CallSigns,
  count(*) FILTER (WHERE CallSign = '') AS WithoutCallSign
FROM Flights
+-------+--------+---------+---------------+
|Flights|Aircraft|CallSigns|WithoutCallSign|
+-------+--------+---------+---------------+
|68371  |29252   |65007    |219            |
+-------+--------+---------+---------------+

